In [37]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv('data.csv')

cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

df_train, df_val = train_test_split(df, test_size=0.2, random_state=42)

num_cols = df.select_dtypes(include=['number']).columns.drop('ID')

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(df_train[num_cols])
X_val_scaled = scaler.transform(df_val[num_cols])

imputer = KNNImputer(n_neighbors=5)
X_train_imputed_scaled = imputer.fit_transform(X_train_scaled)
X_val_imputed_scaled = imputer.transform(X_val_scaled)

X_train_imputed = scaler.inverse_transform(X_train_imputed_scaled)
X_val_imputed = scaler.inverse_transform(X_val_imputed_scaled)

df_train_imputed = pd.DataFrame(X_train_imputed, columns=num_cols, index=df_train.index)
df_val_imputed = pd.DataFrame(X_val_imputed, columns=num_cols, index=df_val.index)

if 'Stage' in num_cols:
    df_train_imputed['Stage'] = df_train_imputed['Stage'].round()
    df_val_imputed['Stage'] = df_val_imputed['Stage'].round()

df_train_final = df_train.copy()
df_val_final = df_val.copy()

for col in num_cols:
    df_train_final[col] = df_train_final[col].fillna(df_train_imputed[col])
    df_val_final[col] = df_val_final[col].fillna(df_val_imputed[col])

df_train_final.to_csv('pbc_train_imputed.csv', index=False)
df_val_final.to_csv('pbc_val_imputed.csv', index=False)

W kolumnach kategorycznych dodaje wartość "unknown" jako osobną klasę wartości. Natomiast dla cech numerycznych używam knn